# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [1]:
from class_dataset import SegmentationClassDataset, num_classes as class_num_classes
from instance_dataset import SegmentationInstanceDataset
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils

import torch

mode = "instance"
# mode = "class"

if mode == "instance":
    dataset_class = SegmentationInstanceDataset
    num_classes = 2
    model_path = "model_instance.pth"
elif mode == "class":
    dataset_class = SegmentationClassDataset
    num_classes = class_num_classes
    model_path = "model_class.pth"

model = get_model_instance_segmentation(num_classes)

dataset = dataset_class("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

{'loss_classifier': tensor(0.8234, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.3860, grad_fn=<DivBackward0>), 'loss_mask': tensor(2.2573, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_objectness': tensor(0.4832, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.0434, grad_fn=<DivBackward0>)}
{'boxes': tensor([[3.7615e+01, 1.3103e+02, 1.5237e+02, 2.9484e+02],
        [1.1628e+02, 0.0000e+00, 3.6634e+02, 2.1164e+02],
        [1.7240e+02, 2.1654e+02, 3.7570e+02, 2.9919e+02],
        [2.4245e+02, 9.7713e+01, 3.5959e+02, 2.9871e+02],
        [4.7437e+01, 2.1279e+02, 2.3245e+02, 3.0000e+02],
        [1.2427e+02, 2.5363e+02, 3.6723e+02, 3.0000e+02],
        [6.4006e+01, 1.7515e+02, 1.6951e+02, 3.0000e+02],
        [9.0972e+00, 2.6055e+02, 2.1085e+02, 3.0000e+02],
        [2.4867e+02, 2.2007e+00, 3.5571e+02, 1.6724e+02],
        [5.4492e+01, 8.4787e+01, 1.8074e+02, 2.7586e+02],
        [2.1986e+02, 1.4560e+02, 3.4161e+02, 2.7695e+02],
        [8

## Training and Validation


In [2]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device("cpu")

# Check for accelerator availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA for training.")
else:
    print("Using CPU for training.")

Using CPU for training.


In [3]:
from engine import evaluate, train_one_epoch


# use our dataset and defined transformations
dataset = dataset_class("../../training_data", get_transforms(is_train=True))
dataset_test = dataset_class("../../training_data", get_transforms(is_train=False))

# split the dataset in train and test set (last 30 images are test set)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-30])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-30:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=utils.collate_fn,
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_test, device=device)

print("That's it!")

# Save the model to disk
torch.save(model.state_dict(), model_path)

Epoch: [0]  [ 0/60]  eta: 0:11:56  lr: 0.000090  loss: 4.2833 (4.2833)  loss_classifier: 0.8245 (0.8245)  loss_box_reg: 0.2820 (0.2820)  loss_mask: 1.1267 (1.1267)  loss_objectness: 1.7061 (1.7061)  loss_rpn_box_reg: 0.3441 (0.3441)  time: 11.9483  data: 1.3479


KeyboardInterrupt: 